# Check out iSnobal outputs at SNOTEL sites

In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyproj

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from pathlib import PurePath
# Locate pyproj_datadir for studio env
# From https://stackoverflow.com/questions/69630630/on-fresh-conda-installation-of-pyproj-pyproj-unable-to-set-database-path-pypr
CONDA_ENV = 'studio'
miniconda_dir = '/uufs/chpc.utah.edu/common/home/u6058223/software/pkg/miniconda3'
proj_version = h.fn_list(miniconda_dir, f'envs/{CONDA_ENV}/conda-meta/proj-[0-9]*.json')[0]

VERSION = PurePath(proj_version).stem
pyprojdatadir = f'{miniconda_dir}/pkgs/{VERSION}/share/proj'
print(pyprojdatadir)
pyproj.datadir.set_data_dir(pyprojdatadir)

In [ ]:
workdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/model_runs/'
script_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts'

In [ ]:
# Basin-specific variables
basin = 'yampa'

# Select just for all variable output of time decay
basindirs = h.fn_list(workdir, f'*{basin}*/wy*/{basin}*/')
_ = [print(b) for b in basindirs]
# Get the WY from the directory name - assumes there is only one WY per basin right now
WYs = [int(basindir.split('wy')[1][:4]) for basindir in basindirs]
WYs = np.unique(WYs)

if len(WYs) > 1:
    print(f'Multiple water years in {basin} basin: {WYs}')
elif len(WYs) == 0:
    print(f'No water years detected in {basin} basin')

bdx = 4
WY = WYs[bdx]
print(f'{WY} selected')

# Adjust basin dirs based on WY
basindirs = h.fn_list(workdir, f'{basin}*/wy{WY}/{basin}*/')

# # Figure out filenames
# poly_fn = '/uufs/chpc.utah.edu/common/home/skiles-group1/arobledano_test/br_100m/br_setup/ancillary/br_basin_outline_HUC8_32612.shp'
# print(poly_fn)
poly_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/ancillary/polys'
poly_fn = h.fn_list(poly_dir, f'*{basin}*shp')[0]
# SNOTEL sites file
# site_locs_fn = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products//SNOTEL/snotel_sites_32612.json'
# epsg = 32612
# state_abbrev = 'AZ'
site_locs_fn = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products//SNOTEL/snotel_sites_32613.json'
epsg = 32613
state_abbrev = 'CO'
# # Locate basin setup dir
# basin_setupdir = '/uufs/chpc.utah.edu/common/home/skiles-group1/arobledano_test/br_100m/br_setup/'
# print(basin_setupdir)

# Note WY dir
wydir = h.fn_list(workdir, f'{basin}*/wy{WY}/')[0]
print(wydir)

In [ ]:
var = 'depth'
snowvar = 'SNOWDEPTH'
thisvar = 'thickness'

# Add for SWE and density as well

In [ ]:
# Locate SNOTEL sites within basin
found_sites = proc.locate_snotel_in_poly(poly_fn=poly_fn, site_locs_fn=site_locs_fn, buffer=200)

# Get site names and site numbers
sitenames = found_sites['site_name']
sitenums = found_sites['site_num']
print(sitenames)

ST_arr = [state_abbrev] * len(sitenums)
gdf_metloom, snotel_dfs, snotel_metadf = proc.get_snotel(sitenums, sitenames, ST_arr, WY=int(WY), epsg=epsg, return_meta=True, snowvar=snowvar)
gdf_metloom

In [ ]:
# Locate csvs
csvs = h.fn_list(wydir, f'*{thisvar}*csv')
df_list = [pd.read_csv(csv, index_col=0) for csv in csvs]
for df in df_list:
    df.index.name = 'Date'
    # Set as DatetimeIndex
    df.index = pd.to_datetime(df.index)
print(csvs)

In [ ]:
ncols = 2
if len(snotel_dfs.keys()) % 2 == 0:
    nrows = len(snotel_dfs.keys()) // 2
else:
    nrows = len(snotel_dfs.keys()) // 2 + 1
print(nrows, ncols)

In [ ]:
# Create labels based on csv filenames
labels = []
colors = []
for csv in csvs:
    # split by basin name
    csv = PurePath(csv).stem
    short_name = csv.split(f'{basin}_')[1]
    # split by variable name
    short_name = short_name.split(f'_{thisvar}_')[0]
    print(short_name)
    if short_name == 'iSnobal-HRRR':
        labels.append('Baseline')
        colors.append('cornflowerblue')
    else:
        labels.append('HRRR-SPIReS')
        colors.append('coral')

In [ ]:
ylims = (0, 3.5)
if basin == 'blue':
    ylims = (0, 2.5)


In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3*nrows), sharex=True, sharey=True)
for kdx, k in enumerate(snotel_dfs.keys()):
    ax = axes.flatten()[kdx]
    snotel_dfs[k]['SNOWDEPTH_m'].plot(label='SNOTEL', ax=ax, c='k', linewidth=1)
    for jdx, label in enumerate(labels):
        df_list[jdx][k].plot(label=label,
                         ax=ax, c=colors[jdx])
    ax.set_ylim(ylims)
    ax.annotate(k, xy=(0.96, 0.84), xycoords='axes fraction', ha='right', fontsize=11, fontstyle='italic')
    ax.grid('--', alpha=0.2)
    ax.legend(bbox_to_anchor=(0.7, 0.7))
# Turn off the last one if that is empty
if len(snotel_dfs.keys()) < len(axes.flatten()):
    ax = axes.flatten()[kdx+1]
    ax.axis('off')
plt.suptitle(f'{basin} WY{WY} SNOTEL comparison with iSnobal {thisvar}', fontsize=12, y=1)

savefig = False
if savefig:
    fig_fn = f'{basin}_wy{WY}_snotelchecks.png'
    fig.savefig(fig_fn, dpi=300, bbox_inches='tight')
plt.tight_layout()